# B2-023-generative-models-diffusion — Practice p24 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** stable-diffusion

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

U = torch.tensor([[0.4, -0.2], [0.0, 1.0]])
C = torch.tensor([[1.0, 0.2], [-0.5, 0.5]])
ATOL = 1e-6
RTOL = 1e-6

## Solution

### (A) Leakage in the conditioning set

The loader's `latent4d` split has 192 training IDs and 48 held-out IDs; line 1–2 load **all** 240 rows and labels, so nothing in the code separates the roles after that.

| edge | lines | held-out information | reaches | class |
|---|---|---|---|---|
| 1 | 3 → 4 → 6 | rows **and labels** of the first 24 held-out IDs (`HELDOUT_IDS[:24]` appended to `cond_ids`) | the training latents `z0` (line 4) and the training call `train_on(model, z0, labels[cond_ids])` (line 6): forward passes, `backward()`, optimizer steps | **data leakage** |
| 2 | 11 → 12 | the 48 held-out labels `labels[heldout][:48]` used as the scoring targets of `hit_rate` | the choice `best_w = max(...)` over $w\in\{0.5,1,1.5\}$ (line 12), a hyperparameter decision | **selection leakage** |
| 3 | 12 → 13 → 14 | the same held-out labels, already used to pick `best_w` and (half of them) to train | the reported "final held-out score" (line 14), computed with the selected `best_w` on the same held-out targets | consequence of both: the report is contaminated by data leakage (edge 1) and selection leakage (edge 2), so it is optimistically biased and is not a held-out estimate |

### (B) Guidance sign

Line **10**, `return e_u - w * (e_c - e_u)`, flips the guidance term; the correct combination is $\varepsilon_u+w(\varepsilon_c-\varepsilon_u)$ (`cfg_combine`).
The two agree only at $w=0$; at $w=1$ the correct one returns $\varepsilon_c$ while the flipped one returns $2\varepsilon_u-\varepsilon_c$.
Because $\varepsilon_c-\varepsilon_u$ points (up to sign and scale) along the gradient of $\log p(c\mid z_t)$, the flipped version moves each sample **away** from its requested class — toward regions where the class is less likely — and larger $w$ pushes harder in the wrong direction.

In [ ]:
def cfg_combine(eps_uncond, eps_cond, w):
    if eps_uncond.shape != eps_cond.shape:
        raise ValueError("eps_uncond and eps_cond must have the same shape")
    return eps_uncond + w * (eps_cond - eps_uncond)


def flipped_guidance(u, c, w):
    return u - w * (c - u)


cfg_w0, flip_w0 = cfg_combine(U, C, 0.0), flipped_guidance(U, C, 0.0)
cfg_w1, flip_w1 = cfg_combine(U, C, 1.0), flipped_guidance(U, C, 1.0)
print("w=0:", cfg_w0.tolist(), flip_w0.tolist())
print("w=1:", cfg_w1.tolist(), flip_w1.tolist())

### (C) Latent scaling factor

The two lines whose scaling disagrees are **line 4**, `z0 = encode(rows[cond_ids])`, which trains on **unscaled** latents (no `* LATENT_SCALE`), and **line 13**, `decode(sample(guide, best_w) / LATENT_SCALE)`, which **divides** by the factor before decoding (line 12 repeats the same decode expression inside the selection).
The cell below shows that raw training latents have per-coordinate standard deviations above $2$ (so the model learns, and its sampler produces, latents at that raw scale), while the scaled latents the schedule was designed for are below $0.7$; dividing raw-scale samples by $0.25$ then multiplies them by $4$, and indeed `decode(encode(C3) / LATENT_SCALE)` is exactly `4 * C3`.

**What the samples look like.** Every decoded sample is blown up by a factor of $4$ about the origin: the class structure (three clusters) survives, but the clusters sit at four times the class centres' positions and are four times as spread out, far from every real data row (and the raw latent scale also leaves $\sqrt{\bar\alpha_T}\,z_0$ not quite negligible at $t=T$).
**Why the training loss gives no warning.** The loss $\lVert\varepsilon-\hat\varepsilon\rVert^2$ is computed on whatever latents line 4 provides and never touches `decode`; a denoiser trained on raw latents fits its ε-targets about as well as one trained on scaled latents, so the loss decreases normally, and the mismatch appears only at decoding, which no training quantity sees.

In [ ]:
E = generative_data.encoder_matrix()
LATENT_SCALE = generative_data.LATENT_SCALE


def encode(x):
    return x @ E.T


def decode(z):
    return z @ E


R = generative_data.train_tensor("latent4d")
raw_std = encode(R).std(dim=0)
scaled_std = (encode(R) * LATENT_SCALE).std(dim=0)
C3 = generative_data.class_centers()
mis_scaled_centres = decode(encode(C3) / LATENT_SCALE)
print("raw latent std   :", [round(v, 4) for v in raw_std.tolist()])
print("scaled latent std:", [round(v, 4) for v in scaled_std.tolist()])
print("decode(encode(C3)/s) / C3 ratio:", (mis_scaled_centres / C3)[C3.abs() > 1e-6].unique().tolist())

### (D) Repair

**Validation carve-out.** The validation subset is carved from the **training IDs only**, stratified by class: for each class $c\in\{0,1,2\}$, the **last 16** training IDs with label $c$ (in stored order). That gives a validation set of $3\times16=48$ IDs and a training-minus-validation set of $192-48=144$ IDs (48 per class); with the loader's stored split these are IDs 180–183, 185–188, 190–193, 195–198, 200–203, 205–208, 210–213, 215–218, 220–223, 225–228, 230–233, 235–238 (printed and checked below). Held-out IDs are not touched until the final step.

**Repaired pipeline (numbered listing).**

```text
 1  train_ids = list(generative_data.TRAIN_IDS["latent4d"])                     # 192 training IDs only
 2  train_labels_all = generative_data.load_labels("latent4d", train_ids)
 3  val_ids = [last 16 training IDs of each class, stored order]                  # 48 IDs, carved from training
 4  fit_ids = [i for i in train_ids if i not in val_ids]                          # 144 IDs
 5  z0 = encode(generative_data.load_rows("latent4d", fit_ids)) * LATENT_SCALE   # scale once, at encoding
 6  torch.manual_seed(20261015); model = LatentDenoiser()
 7  train_on(model, z0, generative_data.load_labels("latent4d", fit_ids))
 8  def guide(z, ts, y, w):
 9      e_u = model(z, ts, y, torch.zeros(len(y), dtype=torch.bool))
10      e_c = model(z, ts, y, torch.ones(len(y), dtype=torch.bool))
11      return cfg_combine(e_u, e_c, w)                                          # correct sign
12  val_labels = generative_data.load_labels("latent4d", val_ids)
13  best_w = max((0.5, 1.0, 1.5), key=lambda w: hit_rate(decode(sample(guide, w) / LATENT_SCALE), val_labels))
14  heldout_labels = generative_data.heldout_label_tensor("latent4d")            # touched once, at the end
15  x = decode(sample(guide, best_w) / LATENT_SCALE)                              # unscale once, before decoding
16  report(hit_rate(x, heldout_labels))                                           # final held-out score
```

Here `sample(guide, w)` generates one sample per entry of the target labels it is evaluated against (validation labels in line 13, held-out labels in line 15). The code below implements lines 1–12 and 14 exactly and defines the remaining steps as functions, so it runs without training; the `encode` recorder is active only while the repaired training latents (line 5) are built.

In [ ]:
DATASET = "latent4d"
train_ids = list(generative_data.TRAIN_IDS[DATASET])
heldout_ids = list(generative_data.HELDOUT_IDS[DATASET])
train_labels_all = generative_data.load_labels(DATASET, train_ids)

val_ids = []
for c in range(3):
    ids_c = [i for i, y in zip(train_ids, train_labels_all.tolist()) if y == c]
    val_ids.extend(ids_c[-16:])
val_ids = sorted(val_ids)
val_set = set(val_ids)
fit_ids = [i for i in train_ids if i not in val_set]

# Build the repaired training latents with the clean-row seam (encode) recorded.
recorded = []
original_encode = encode


def recording_encode(x):
    recorded.append(x.detach().clone())
    return original_encode(x)


encode = recording_encode
try:
    z0_repaired = encode(generative_data.load_rows(DATASET, fit_ids)) * LATENT_SCALE
finally:
    encode = original_encode
fit_labels = generative_data.load_labels(DATASET, fit_ids)
val_labels = generative_data.load_labels(DATASET, val_ids)

hashes = generative_data.ROW_SHA256[DATASET]
heldout_hashes = {hashes[i] for i in heldout_ids}
fit_hashes = {hashes[i] for i in fit_ids}
recorded_rows = torch.cat(recorded)
recorded_hashes = [generative_data.row_sha256(row) for row in recorded_rows]


def hit_rate(x, y):
    centers = generative_data.class_centers()
    nearest = ((x[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2).argmin(dim=1)
    return (nearest == y).float().mean().item()


def repaired_selection_and_report(model, sample):
    """Lines 8-16: correct guidance sign, w chosen on validation labels, held-out used once at the end."""
    def guide(z, ts, y, w):
        e_u = model(z, ts, y, torch.zeros(len(y), dtype=torch.bool))
        e_c = model(z, ts, y, torch.ones(len(y), dtype=torch.bool))
        return cfg_combine(e_u, e_c, w)

    best_w = max((0.5, 1.0, 1.5), key=lambda w: hit_rate(decode(sample(guide, w, val_labels) / LATENT_SCALE), val_labels))
    heldout_labels = generative_data.heldout_label_tensor(DATASET)
    x = decode(sample(guide, best_w, heldout_labels) / LATENT_SCALE)
    return best_w, hit_rate(x, heldout_labels)


print("validation IDs (48):", val_ids)
print("sizes: train-minus-validation", len(fit_ids), " validation", len(val_ids), " held-out", len(heldout_ids))
print("per-class fit counts:", torch.bincount(fit_labels, minlength=3).tolist(),
      " per-class validation counts:", torch.bincount(val_labels, minlength=3).tolist())
print("repaired training latents:", tuple(z0_repaired.shape), " scaled std:", [round(v, 4) for v in z0_repaired.std(dim=0).tolist()])

### Answer check

In [ ]:
# (B) guidance probes
assert torch.equal(cfg_w0, flip_w0)
assert torch.allclose(cfg_w1, C, atol=ATOL, rtol=RTOL)
assert torch.allclose(flip_w1, torch.tensor([[-0.2, -0.6], [0.5, 1.5]]), atol=ATOL, rtol=RTOL)
# (C) scaling probes
assert bool((raw_std > 2.0).all())
assert bool((scaled_std < 0.7).all())
assert torch.allclose(mis_scaled_centres, 4 * C3, atol=ATOL, rtol=RTOL)
# (D) split bookkeeping
assert len(train_ids) == 192 and len(heldout_ids) == 48
assert len(val_ids) == 48 and len(fit_ids) == 144
assert val_set <= set(train_ids) and not (val_set & set(fit_ids)) and not (set(train_ids) & set(heldout_ids))
assert torch.bincount(val_labels, minlength=3).tolist() == [16, 16, 16]
assert val_ids == [i for start in range(180, 240, 5) for i in range(start, start + 4)]   # the IDs stated above
# (D) hash check at the encode seam
assert len(recorded) == 1
assert not (set(recorded_hashes) & heldout_hashes)
assert set(recorded_hashes) == fit_hashes and len(recorded_hashes) == len(fit_ids)
# (D) repaired training latents: (train-minus-validation size, 2)
assert tuple(z0_repaired.shape) == (144, 2)